In [1]:
#Kopiere inn filer fra USB
!cp /media/ubuntu/KINGSTON/Testing-HLS4ML/* ./
!ls
!pwd

cp: cannot stat '/media/ubuntu/KINGSTON/Testing-HLS4ML/*': No such file or directory
Playground			 X_train_val.npy	     classes.npy
Playground-testing-HLS4ML.ipynb  blokkdiagram_1_wrapper.bit  y_test.npy
X_test.npy			 blokkdiagram_1_wrapper.hwh  y_train_val.npy
/root/jupyter_notebooks/Testing


## Laste bitfil
og ulike kommandoer for å undersøke blokkdiagrammet i PL

In [9]:
# Får error når jeg bruker kv260-biblioteket
#from kv260 import BaseOverlay

#blokkdiagram = BaseOverlay("blokkdiagram_1_wrapper.bit")

In [15]:
# Denne fungerer. Hentet fra resizer_pl
from pynq import allocate, Overlay
blokkdiagram = Overlay("blokkdiagram_1_wrapper.bit")

In [16]:
#blokkdiagram?

In [17]:
#blokkdiagram.download()

In [18]:
#blokkdiagram.timestamp

In [19]:
#blokkdiagram.is_loaded()

velger dma

In [20]:
dma = blokkdiagram.axi_dma_0
#nn = blokkdiagram.testmodel_1_hls4ml

In [21]:
dma.register_map

RegisterMap {
  MM2S_DMACR = Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  MM2S_DMASR = Register(Halted=1, Idle=0, SGIncld=1, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SGDecErr=0, IOC_Irq=0, Dly_Irq=0, Err_Irq=0, IRQThresholdSts=1, IRQDelaySts=0),
  MM2S_CURDESC = Register(Current_Descriptor_Pointer=0),
  MM2S_CURDESC_MSB = Register(Current_Descriptor_Pointer=0),
  MM2S_TAILDESC = Register(Tail_Descriptor_Pointer=0),
  MM2S_TAILDESC_MSB = Register(Tail_Descriptor_Pointer=0),
  MM2S_SA = Register(Source_Address=0),
  MM2S_SA_MSB = Register(Source_Address=0),
  MM2S_LENGTH = Register(Length=0),
  SG_CTL = Register(SG_CACHE=3, SG_USER=0),
  S2MM_DMACR = Register(RS=0, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  S2MM_DMASR = Register(Halted=1, Idle=0, SGIncld=1, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SG

# Inferens
Baserer seg på part7b fra hls4ml-tutorial. Den bruker VivadoAccelerator-backend for å bygge hele blokkdiagrammet, mens her testes manuelt bygde.

In [22]:
# Laste testdata inn i mappa
#!cp /home/ubuntu/HLS4ML/hls4ml-tutorial/*.npy .
#!ls

In [23]:
import numpy as np
x_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

In [24]:
print(x_test.shape,y_test.shape)

(166000, 16) (166000, 5)


In [25]:
x_test[0]

array([-0.11950312,  0.40616292, -1.04058613, -0.82473132, -0.75533621,
       -0.58097338,  1.98691601,  1.53806275,  1.98691601,  0.6312378 ,
        0.38356148, -0.20130531,  1.05976638,  0.40865762, -1.01995561,
       -0.18016747])

In [26]:
x_test.dtype # ups, blokken tar vel bare i mot float 16

dtype('float64')

In [27]:
x_test = x_test.astype(np.float16)
x_test.dtype

dtype('float16')

In [28]:
x_test[0]

array([-0.1195,  0.4062, -1.041 , -0.8247, -0.7554, -0.581 ,  1.987 ,
        1.538 ,  1.987 ,  0.6313,  0.3835, -0.2013,  1.06  ,  0.4087,
       -1.02  , -0.1802], dtype=float16)

Basert på kildekoden til VivadoAccelerators NeuralNetworkOverlay-klasse som er et slags PYNQ-tranlation layer for Python i PS til PL.

In [29]:
input_shape = x_test.shape
#input_shape = (,)

output_shape = y_test.shape
#output_shape = (80,) # vet at output er 80 bit = 5*16bit, så bare kjører det nå

In [30]:
in_buffer = allocate(shape=input_shape,
                     dtype=np.float16, 
                     cacheable=1)
out_buffer = allocate(shape=output_shape, 
                      dtype=np.float16, 
                      cacheable=1)

In [31]:
in_buffer[:] = x_test#[0:1]
#in_buffer[:] = np.zeros(x_test.shape)

In [32]:
import time

def run_inferens():
    dma.sendchannel.transfer(in_buffer)
    dma.recvchannel.transfer(out_buffer)    
    #resizer.write(0x00,0x81) # start
    #dma.sendchannel.wait()
    #dma.recvchannel.wait()
    time.sleep(1)

In [33]:
#resizer.register_map.src_rows = old_height
#resizer.register_map.src_cols = old_width
#resizer.register_map.dst_rows = new_height
#resizer.register_map.dst_cols = new_width

In [34]:
run_inferens()

In [35]:
pred_resultat = out_buffer
pred_resultat

PynqBuffer([[1.162e-05, 5.960e-08, 6.104e-05, 0.000e+00, 0.000e+00],
            [0.000e+00, 0.000e+00, 9.000e-06, 5.960e-06, 4.882e-05],
            [0.000e+00, 0.000e+00, 8.762e-06, 5.245e-06, 4.882e-05],
            ...,
            [0.000e+00, 0.000e+00, 0.000e+00, 6.104e-05, 1.132e-06],
            [0.000e+00, 0.000e+00, 0.000e+00, 6.104e-05, 2.742e-06],
            [1.270e-05, 1.192e-07, 4.882e-05, 0.000e+00, 0.000e+00]],
           dtype=float16)

In [36]:

#np.argmax(pred_resultat,axis=1)

PynqBuffer([1.162e-05, 5.960e-08, 6.104e-05, 0.000e+00, 0.000e+00],
           dtype=float16)

In [111]:
# Resette DMA som en hotfix for at jeg ikke har implementert DMAen/greiene riktig i blokkdiagrammet
#del dma
#dma.sendchannel.stop()
#må bare laste ned hele greia på nytt

KeyboardInterrupt: 

In [47]:
y_test[0]

array([0., 1., 0., 0., 0.], dtype=float32)

In [46]:
import struct
#''.join('{:0>8b}'.format(c))
s = struct.pack('>f',pred_resultat[0][0])
struct.unpack('>l',s)[0]
hex(_)

'0x37430000'